In [ ]:
!pip install -q fastapi uvicorn[standard] pyngrok httpx "diffusers>=0.27.0" transformers accelerate safetensors
!pip install -q xformers  # if it fails, you can skip; just a speedup
!pip install -q imageio imageio-ffmpeg



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.9 MB/s eta 0:00:00


In [ ]:
import os
import torch, uvicorn, threading, tempfile
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, FileResponse
from pyngrok import ngrok
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video, load_image

# ---------- CONFIG ----------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VIDEO_DIR = "/content/generated_videos"
os.makedirs(VIDEO_DIR, exist_ok=True)

# target duration ~10 seconds with 25 frames:
EXPORT_FPS = 2.5  # 25 frames / 2.5 fps = 10 seconds

# ---------- MODEL LOAD ----------
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",  # 25 frames model
    torch_dtype=torch.float16,
    variant="fp16",
)

if DEVICE == "cuda":
    pipe.to("cuda")
    pipe.enable_model_cpu_offload()
else:
    pipe.to("cpu")

# ---------- FASTAPI APP ----------
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # or ["http://127.0.0.1:5500"]
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/api/image2video")
async def image_to_video(
    prompt: str = Form(""),
    quality: str = Form("good"),
    image: UploadFile = File(...),
):
    # for this model we keep num_frames at 25 for best quality
    num_frames = 25

    # save uploaded image to temp file
    img_bytes = await image.read()
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        tmp.write(img_bytes)
        tmp_path = tmp.name

    try:
        init_image = load_image(tmp_path).convert("RGB")

        height, width = 480, 720

        with torch.inference_mode():
            result = pipe(
                init_image,
                num_frames=num_frames,
                height=height,
                width=width,
                decode_chunk_size=4,
            )

        frames = result.frames[0]  # list of PIL frames

        # save video to a persistent mp4 file (~10 s at 2.5 fps)
        video_filename = next(tempfile._get_candidate_names()) + ".mp4"
        video_path = os.path.join(VIDEO_DIR, video_filename)
        export_to_video(frames, video_path, fps=EXPORT_FPS)

        # optional: print size to confirm it's not empty
        print("Saved video:", video_path, "bytes:", os.path.getsize(video_path))

    except Exception as e:
        print("SVD ERROR:", repr(e))
        return JSONResponse(
            status_code=500,
            content={"error": f"SVD generation failed: {str(e)}"},
        )

    return {"video_url": f"/video/{video_filename}"}

@app.get("/video/{filename}")
def get_video(filename: str):
    file_path = os.path.join(VIDEO_DIR, filename)
    if not os.path.isfile(file_path):
        return JSONResponse(status_code=404, content={"error": "Video not found"})
    return FileResponse(file_path, media_type="video/mp4")

# ---------- UVICORN + NGROK ----------
def run():
    uvicorn.run(app, host="0.0.0.0", port=8004)

thread = threading.Thread(target=run, daemon=True)
thread.start()

ngrok.set_auth_token("37OPKAeuYozMBZMbMgHLInFJmNj_31oRB8XiwNwZCkUfqq5Qg")
public_url_i2v = ngrok.connect(8004, "http").public_url
public_url_i2v


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/496 [00:00<?, ?B/s]

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

INFO:     Started server process [458]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8004 (Press CTRL+C to quit)


'https://encephalographic-leena-unrepulsing.ngrok-free.dev'